In [7]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import goatools
from tqdm import tqdm

# BioMaRt mapping files

In [2]:
from pybiomart import Server

server = Server(host='http://www.ensembl.org')

dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
                 .datasets['hsapiens_gene_ensembl'])

In [3]:
ensembl_gene_uniprot = dataset.query(attributes=['ensembl_gene_id', 'uniprotswissprot'])
ensembl_gene_uniprot = ensembl_gene_uniprot[~ensembl_gene_uniprot["UniProtKB/Swiss-Prot ID"].isnull()]
ensembl_gene_uniprot

,Gene stable ID,UniProtKB/Swiss-Prot ID
0,ENSG00000198888,P03886
1,ENSG00000198763,P03891
2,ENSG00000198804,P00395
3,ENSG00000198712,P00403
4,ENSG00000228253,P03928
...,...,...
38143,ENSG00000142961,Q70IA8
38144,ENSG00000116754,Q05519
38146,ENSG00000283324,A0A1B0GV90
38148,ENSG00000077585,O60478


In [4]:
ensembl_transcript_uniprot = dataset.query(attributes=['ensembl_transcript_id', 'uniprotswissprot'])
ensembl_transcript_uniprot = ensembl_transcript_uniprot[~ensembl_transcript_uniprot["UniProtKB/Swiss-Prot ID"].isnull()]
ensembl_transcript_uniprot

,Transcript stable ID,UniProtKB/Swiss-Prot ID
0,ENST00000361390,P03886
1,ENST00000361453,P03891
2,ENST00000361624,P00395
3,ENST00000361739,P00403
4,ENST00000361851,P03928
...,...,...
117913,ENST00000370951,Q05519
117915,ENST00000370950,Q05519
117921,ENST00000636087,A0A1B0GV90
117923,ENST00000366592,O60478


In [5]:
ensembl_gene_rnacentral = dataset.query(attributes=['ensembl_gene_id', 'rnacentral'])
ensembl_gene_rnacentral = ensembl_gene_rnacentral[~ensembl_gene_rnacentral["RNAcentral ID"].isnull()]
ensembl_gene_rnacentral

,Gene stable ID,RNAcentral ID
0,ENSG00000210049,URS00003E8921
1,ENSG00000211459,URS000044DFF6
2,ENSG00000210077,URS00002D2D8F
3,ENSG00000210082,URS000047A7F4
4,ENSG00000209082,URS000061A10B
...,...,...
130084,ENSG00000286142,URS0000EEB0E2
130086,ENSG00000077585,URS0000D27E7B
130087,ENSG00000077585,URS00008C3E98
130088,ENSG00000222650,URS000064A434


In [6]:
ensembl_transcript_rnacentral = dataset.query(attributes=['ensembl_transcript_id', 'rnacentral'])
ensembl_transcript_rnacentral = ensembl_transcript_rnacentral[~ensembl_transcript_rnacentral["RNAcentral ID"].isnull()]
ensembl_transcript_rnacentral

,Transcript stable ID,RNAcentral ID
0,ENST00000387314,URS00003E8921
1,ENST00000389680,URS000044DFF6
2,ENST00000387342,URS00002D2D8F
3,ENST00000387347,URS000047A7F4
4,ENST00000386347,URS000061A10B
...,...,...
266595,ENST00000650982,URS0000EEB0E2
266598,ENST00000366591,URS0000D27E7B
266600,ENST00000477559,URS00008C3E98
266601,ENST00000410718,URS000064A434


# Settings

In [7]:
goa_human = "/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human.gaf"
goa_human_isoform = "/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_isoform.gaf"
goa_human_rna = "/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_rna.gaf"
gene_transcript = "gene"
uniprot_or_rnacentral = "uniprot"
#evidence_codes = ['EXP', 'IDA', 'IPI', 'IMP', 'IGI', 'IEP', 'HTP', 'HDA', 'HMP', 'HGI', 'HEP', 'IBA', 'IBD', 'IKR', 'IRD', 'TAS', 'NAS', 'IC', 'ND']
evidence_codes = None
columns = ["DB", "DB_Object_ID", "DB_Object_Symbol", "Qualifier", "GO_ID", "DB_Reference", "Evidence_Code", "With_or_From", "Aspect", "DB_Object_Name", "DB_Object_Synonym", "DB_Object_Type", "Taxon", "Date", "Assigned_by", "Annotation_Extension", "Annotation_Properties"]
databases = ["UniProt", "Reactome", "IntAct", "Ensembl", "GO_Central"]
#datbases = None

# GAF file creation

## Original GOA files

### Load original GAF

In [9]:
#gaf = pd.read_csv(goa_human_isoform, delimiter="\t", header=None, skiprows=27)
gaf = pd.read_csv(goa_human, delimiter="\t", header=None, skiprows=41)
#gaf = pd.read_csv(goa_human_rna, delimiter="\t", header=None, skiprows=23)
gaf.columns = columns
gaf

/data/analysis/ag-reils/ag-reils-shared/deps/miniconda3/envs/u006/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (15) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_or_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_by,Annotation_Extension,Annotation_Properties
0,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0003723,GO_REF:0000043,IEA,UniProtKB-KW:KW-0694,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
1,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0046872,GO_REF:0000043,IEA,UniProtKB-KW:KW-0479,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
2,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052840,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
3,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052842,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
4,UniProtKB,A0A024RBG1,NUDT4B,located_in,GO:0005829,GO_REF:0000052,IDA,NaN,C,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20161204,HPA,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617523,UniProtKB,Q9UM73,ALK,involved_in,GO:0042127,PMID:21873635,IBA,PANTHER:PTN000699408|MGI:MGI:103305,P,ALK tyrosine kinase receptor,ALK,protein,taxon:9606,20181026,GO_Central,NaN,NaN
617524,UniProtKB,Q9H4H8,FAM83D,involved_in,GO:0007165,PMID:21873635,IBA,PANTHER:PTN004490657|UniProtKB:Q5T0W9|UniProtK...,P,Protein FAM83D,FAM83D|C20orf129,protein,taxon:9606,20200408,GO_Central,NaN,NaN
617525,UniProtKB,Q9BXS6,NUSAP1,is_active_in,GO:0005730,PMID:21873635,IBA,PANTHER:PTN001286938|MGI:MGI:2675669,C,Nucleolar and spindle-associated protein 1,NUSAP1|ANKT|BM-037|PRO0310,protein,taxon:9606,20170228,GO_Central,NaN,NaN
617526,UniProtKB,Q8NHG7,SVIP,involved_in,GO:1904153,PMID:21873635,IBA,PANTHER:PTN002173988|UniProtKB:Q8NHG7,P,Small VCP/p97-interacting protein,SVIP,protein,taxon:9606,20200702,GO_Central,NaN,NaN


### Modify GAF + Map to genes/ transcripts

In [231]:
if uniprot_or_rnacentral == "uniprot":
    if databases:
        gaf = gaf[gaf["Assigned_by"].isin(databases)]
gaf

,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_or_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_by,Annotation_Extension,Annotation_Properties
0,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0003723,GO_REF:0000043,IEA,UniProtKB-KW:KW-0694,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
1,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0046872,GO_REF:0000043,IEA,UniProtKB-KW:KW-0479,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
2,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052840,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
3,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052842,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
5,UniProtKB,A0A075B6H7,IGKV3-7,involved_in,GO:0002250,GO_REF:0000043,IEA,UniProtKB-KW:KW-1064,P,Probable non-functional immunoglobulin kappa v...,IGKV3-7,protein,taxon:9606,20211010,UniProt,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617523,UniProtKB,Q9UM73,ALK,involved_in,GO:0042127,PMID:21873635,IBA,PANTHER:PTN000699408|MGI:MGI:103305,P,ALK tyrosine kinase receptor,ALK,protein,taxon:9606,20181026,GO_Central,NaN,NaN
617524,UniProtKB,Q9H4H8,FAM83D,involved_in,GO:0007165,PMID:21873635,IBA,PANTHER:PTN004490657|UniProtKB:Q5T0W9|UniProtK...,P,Protein FAM83D,FAM83D|C20orf129,protein,taxon:9606,20200408,GO_Central,NaN,NaN
617525,UniProtKB,Q9BXS6,NUSAP1,is_active_in,GO:0005730,PMID:21873635,IBA,PANTHER:PTN001286938|MGI:MGI:2675669,C,Nucleolar and spindle-associated protein 1,NUSAP1|ANKT|BM-037|PRO0310,protein,taxon:9606,20170228,GO_Central,NaN,NaN
617526,UniProtKB,Q8NHG7,SVIP,involved_in,GO:1904153,PMID:21873635,IBA,PANTHER:PTN002173988|UniProtKB:Q8NHG7,P,Small VCP/p97-interacting protein,SVIP,protein,taxon:9606,20200702,GO_Central,NaN,NaN


In [232]:
if evidence_codes:
    gaf = gaf[gaf["Evidence_Code"].isin(evidence_codes)]
gaf

,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_or_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_by,Annotation_Extension,Annotation_Properties
0,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0003723,GO_REF:0000043,IEA,UniProtKB-KW:KW-0694,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
1,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0046872,GO_REF:0000043,IEA,UniProtKB-KW:KW-0479,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
2,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052840,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
3,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0052842,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
5,UniProtKB,A0A075B6H7,IGKV3-7,involved_in,GO:0002250,GO_REF:0000043,IEA,UniProtKB-KW:KW-1064,P,Probable non-functional immunoglobulin kappa v...,IGKV3-7,protein,taxon:9606,20211010,UniProt,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617523,UniProtKB,Q9UM73,ALK,involved_in,GO:0042127,PMID:21873635,IBA,PANTHER:PTN000699408|MGI:MGI:103305,P,ALK tyrosine kinase receptor,ALK,protein,taxon:9606,20181026,GO_Central,NaN,NaN
617524,UniProtKB,Q9H4H8,FAM83D,involved_in,GO:0007165,PMID:21873635,IBA,PANTHER:PTN004490657|UniProtKB:Q5T0W9|UniProtK...,P,Protein FAM83D,FAM83D|C20orf129,protein,taxon:9606,20200408,GO_Central,NaN,NaN
617525,UniProtKB,Q9BXS6,NUSAP1,is_active_in,GO:0005730,PMID:21873635,IBA,PANTHER:PTN001286938|MGI:MGI:2675669,C,Nucleolar and spindle-associated protein 1,NUSAP1|ANKT|BM-037|PRO0310,protein,taxon:9606,20170228,GO_Central,NaN,NaN
617526,UniProtKB,Q8NHG7,SVIP,involved_in,GO:1904153,PMID:21873635,IBA,PANTHER:PTN002173988|UniProtKB:Q8NHG7,P,Small VCP/p97-interacting protein,SVIP,protein,taxon:9606,20200702,GO_Central,NaN,NaN


In [233]:
if uniprot_or_rnacentral == "uniprot":
    if gene_transcript == "gene":
        gaf_joined = gaf.join(ensembl_gene_uniprot.set_index("UniProtKB/Swiss-Prot ID"), on="DB_Object_ID")
        gaf_joined["DB_Object_ID"] = gaf_joined["Gene stable ID"]
        del gaf_joined["Gene stable ID"]
        gaf_joined = gaf_joined[~gaf_joined["DB_Object_ID"].isnull()]
    if gene_transcript == "transcript":
        gaf_joined = gaf.join(ensembl_transcript_uniprot.set_index("UniProtKB/Swiss-Prot ID"), on="DB_Object_ID")
        gaf_joined["DB_Object_ID"] = gaf_joined["Transcript stable ID"]
        del gaf_joined["Transcript stable ID"]
        gaf_joined = gaf_joined[~gaf_joined["DB_Object_ID"].isnull()]
elif uniprot_or_rnacentral == "rnacentral":
    gaf["DB_Object_ID"] = gaf["DB_Object_ID"].apply(lambda x: x.split("_")[0])
    if gene_transcript == "gene":
        gaf_joined = gaf.join(ensembl_gene_rnacentral.set_index("RNAcentral ID"), on="DB_Object_ID")
        gaf_joined["DB_Object_ID"] = gaf_joined["Gene stable ID"]
        del gaf_joined["Gene stable ID"]
        gaf_joined = gaf_joined[~gaf_joined["DB_Object_ID"].isnull()]
    if gene_transcript == "transcript":
        gaf_joined = gaf.join(ensembl_transcript_rnacentral.set_index("RNAcentral ID"), on="DB_Object_ID")
        gaf_joined["DB_Object_ID"] = gaf_joined["Transcript stable ID"]
        del gaf_joined["Transcript stable ID"]
        gaf_joined = gaf_joined[~gaf_joined["DB_Object_ID"].isnull()]
gaf_joined

,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_or_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_by,Annotation_Extension,Annotation_Properties
0,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0003723,GO_REF:0000043,IEA,UniProtKB-KW:KW-0694,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
1,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0046872,GO_REF:0000043,IEA,UniProtKB-KW:KW-0479,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
2,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0052840,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
3,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0052842,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
5,UniProtKB,ENSG00000282310,IGKV3-7,involved_in,GO:0002250,GO_REF:0000043,IEA,UniProtKB-KW:KW-1064,P,Probable non-functional immunoglobulin kappa v...,IGKV3-7,protein,taxon:9606,20211010,UniProt,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617523,UniProtKB,ENSG00000171094,ALK,involved_in,GO:0042127,PMID:21873635,IBA,PANTHER:PTN000699408|MGI:MGI:103305,P,ALK tyrosine kinase receptor,ALK,protein,taxon:9606,20181026,GO_Central,NaN,NaN
617524,UniProtKB,ENSG00000101447,FAM83D,involved_in,GO:0007165,PMID:21873635,IBA,PANTHER:PTN004490657|UniProtKB:Q5T0W9|UniProtK...,P,Protein FAM83D,FAM83D|C20orf129,protein,taxon:9606,20200408,GO_Central,NaN,NaN
617525,UniProtKB,ENSG00000137804,NUSAP1,is_active_in,GO:0005730,PMID:21873635,IBA,PANTHER:PTN001286938|MGI:MGI:2675669,C,Nucleolar and spindle-associated protein 1,NUSAP1|ANKT|BM-037|PRO0310,protein,taxon:9606,20170228,GO_Central,NaN,NaN
617526,UniProtKB,ENSG00000198168,SVIP,involved_in,GO:1904153,PMID:21873635,IBA,PANTHER:PTN002173988|UniProtKB:Q8NHG7,P,Small VCP/p97-interacting protein,SVIP,protein,taxon:9606,20200702,GO_Central,NaN,NaN


### Save file

In [234]:
from itertools import islice
#with open(goa_human_isoform, "r") as myfile:
#    head = list(islice(myfile, 27))
with open(goa_human, "r") as myfile:
    head = list(islice(myfile, 41))
#with open(goa_human_rna, "r") as myfile:
#    head = list(islice(myfile, 23))
head

['!gaf-version: 2.2\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T16:45\n',
 '!\n',
 '!Header from source association file:\n',
 '!=================================\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T04:41\n',
 '!\n',
 '!Header from goa_human source association file:\n',
 '!=================================\n',
 '!\n',
 '!The set of protein accessions included in this file is based on UniProt reference proteomes, which provide one protein per gene.\n',
 '!They include the protein sequences annotated in Swiss-Prot or the longest TrEMBL transcript if there is no Swiss-Prot record.\n',
 '!If a particular protein accession is not annotated with GO, then it will not appear in this file.\n',
 '!\n',
 '!Note that the annotation set in this file is filtered in order to reduce redundancy; the full, unfiltered set can be found in\n',
 '!ftp://ftp.ebi.ac.uk/pub/databases/GO/goa/UNIPROT/goa_uniprot_all.gz\n',
 '!\n',
 '!date-gene

In [235]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_databases_uniprot_reactome_ensembl_intact_gocentral_ensembl_gene_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [236]:
gaf_joined.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_databases_uniprot_reactome_ensembl_intact_gocentral_ensembl_gene_mapping.gaf', mode="a", sep="\t", header=False, index=False)

## RNA DK using NPInterv4 database

### Load obodag

In [115]:
from goatools.obo_parser import GODag

obodag = GODag("/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/go-basic.obo")
len(obodag.keys())

/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/go-basic.obo: fmt(1.2) rel(2019-07-01) 47,413 GO Terms


47413

### Load GAF files

In [116]:
gaf_human = pd.read_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_ensembl_gene_mapping.gaf', skiprows=41, header=None, delimiter="\t")
gaf_human_isoform = pd.read_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_isoform_ensembl_gene_mapping.gaf', skiprows=27, header=None, delimiter="\t")
gaf_human_rna = pd.read_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_rna_ensembl_gene_mapping.gaf', skiprows=23, header=None, delimiter="\t")
gaf = pd.concat([gaf_human, gaf_human_isoform])#, gaf_human_rna])
gaf

/data/analysis/ag-reils/ag-reils-shared/deps/miniconda3/envs/u006/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (15) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0003723,GO_REF:0000043,IEA,UniProtKB-KW:KW-0694,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
1,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0046872,GO_REF:0000043,IEA,UniProtKB-KW:KW-0479,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211010,UniProt,NaN,NaN
2,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0052840,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
3,UniProtKB,ENSG00000177144,NUDT4B,enables,GO:0052842,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20211009,UniProt,NaN,NaN
4,UniProtKB,ENSG00000177144,NUDT4B,located_in,GO:0005829,GO_REF:0000052,IDA,NaN,C,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20161204,HPA,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35506,UniProtKB,ENSG00000152217,SETBP1,enables,GO:0005515,PMID:32296183,IPI,UniProtKB:Q7KZS0,F,SET-binding protein,SETBP1|KIAA0437,protein,taxon:9606,20211030,IntAct,NaN,UniProtKB:Q9Y6X0-2
35507,UniProtKB,ENSG00000152217,SETBP1,enables,GO:0005515,PMID:32296183,IPI,UniProtKB:Q9BXN6,F,SET-binding protein,SETBP1|KIAA0437,protein,taxon:9606,20211030,IntAct,NaN,UniProtKB:Q9Y6X0-2
35508,UniProtKB,ENSG00000152217,SETBP1,enables,GO:0005515,PMID:32296183,IPI,UniProtKB:Q9H8W4,F,SET-binding protein,SETBP1|KIAA0437,protein,taxon:9606,20211030,IntAct,NaN,UniProtKB:Q9Y6X0-2
35509,UniProtKB,ENSG00000152217,SETBP1,enables,GO:0005515,PMID:32296183,IPI,UniProtKB:Q9NS26,F,SET-binding protein,SETBP1|KIAA0437,protein,taxon:9606,20211030,IntAct,NaN,UniProtKB:Q9Y6X0-2


### Load NPInterv4

In [117]:
file_npinterv4 = "/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/interaction_NPInterv4.txt"
npinterv4 = pd.read_csv(file_npinterv4, sep="\t", dtype=str)
npinterv4 = npinterv4[npinterv4['organism']=="Homo sapiens"]
npinterv4 = npinterv4[~(npinterv4['tarID']=="-")]
npinterv4 = npinterv4[~(npinterv4['ncID']=="-")]
npinterv4 = npinterv4[npinterv4['tarID'].notna()]
npinterv4 = npinterv4[npinterv4['ncID'].notna()]
npinterv4

,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,reference,organism,tissueOrCell,tag,class,level,datasource
3,ncRI-40000004,7SK,NONHSAG040596,lncRNA,MYC,NM_002467,protein,Electrophoretic mobility shift experiment indi...,EMSA,9018369,Homo sapiens,-,regulatory;promoter as action site,regulatory,RNA-RNA,Literature mining
4,ncRI-40000005,7SK,NONHSAG040596,lncRNA,ABO,NM_020469,protein,Promoters pU6 and p7SK proved to express high ...,-,22522162,Homo sapiens,-,regulatory,regulatory,RNA-RNA,Literature mining
5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,15201869;15994294;17671421;20675720;20926576;1...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining
6,ncRI-40000007,7SK,NONHSAG040596,lncRNA,Oct1,P20263,protein,PTF and Oct-1 enhance transcription from the 7...,-,1535687,Homo sapiens,-,ncRNA is regulated,regulatory,DNA-TF,Literature mining
7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,20976203;20471949,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100531,ncRI-41100533,hsa-miR-151b,MI0003772,miRNA,NUFIP2,ENSG00000108256,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100532,ncRI-41100534,hsa-miR-151b,MI0003772,miRNA,SLC15A4,ENSG00000139370,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100533,ncRI-41100535,hsa-miR-151b,MI0003772,miRNA,TBC1D10B,ENSG00000169221,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100534,ncRI-41100536,hsa-miR-548d-5p,MI0003671,miRNA,HDLBP,ENSG00000115677,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database


In [118]:
npinterv4['ncID'] = npinterv4['ncID'].str.split(';')
npinterv4 = npinterv4.explode('ncID')
npinterv4['tarID'] = npinterv4['tarID'].str.split(';')
npinterv4 = npinterv4.explode('tarID')
npinterv4

,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,reference,organism,tissueOrCell,tag,class,level,datasource
3,ncRI-40000004,7SK,NONHSAG040596,lncRNA,MYC,NM_002467,protein,Electrophoretic mobility shift experiment indi...,EMSA,9018369,Homo sapiens,-,regulatory;promoter as action site,regulatory,RNA-RNA,Literature mining
4,ncRI-40000005,7SK,NONHSAG040596,lncRNA,ABO,NM_020469,protein,Promoters pU6 and p7SK proved to express high ...,-,22522162,Homo sapiens,-,regulatory,regulatory,RNA-RNA,Literature mining
5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,15201869;15994294;17671421;20675720;20926576;1...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining
6,ncRI-40000007,7SK,NONHSAG040596,lncRNA,Oct1,P20263,protein,PTF and Oct-1 enhance transcription from the 7...,-,1535687,Homo sapiens,-,ncRNA is regulated,regulatory,DNA-TF,Literature mining
7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,20976203;20471949,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100531,ncRI-41100533,hsa-miR-151b,MI0003772,miRNA,NUFIP2,ENSG00000108256,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100532,ncRI-41100534,hsa-miR-151b,MI0003772,miRNA,SLC15A4,ENSG00000139370,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100533,ncRI-41100535,hsa-miR-151b,MI0003772,miRNA,TBC1D10B,ENSG00000169221,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database
1100534,ncRI-41100536,hsa-miR-548d-5p,MI0003671,miRNA,HDLBP,ENSG00000115677,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database


### BioMaRt mapping files

#### RNACentral to ENSG

In [119]:
from pybiomart import Server

server = Server(host='http://www.ensembl.org')

dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
                 .datasets['hsapiens_gene_ensembl'])

In [120]:
ensembl_gene_gene_name = dataset.query(attributes=['ensembl_gene_id', 'external_gene_name'])
ensembl_gene_gene_name["Gene name"] = ensembl_gene_gene_name["Gene name"].astype("str")
ensembl_gene_gene_name

,Gene stable ID,Gene name
0,ENSG00000210049,MT-TF
1,ENSG00000211459,MT-RNR1
2,ENSG00000210077,MT-TV
3,ENSG00000210082,MT-RNR2
4,ENSG00000209082,MT-TL1
...,...,...
68000,ENSG00000286142,nan
68001,ENSG00000077585,GPR137B
68002,ENSG00000222650,RNU2-70P
68003,ENSG00000237922,nan


In [121]:
ensembl_gene_uniprot = dataset.query(attributes=['ensembl_gene_id', 'uniprotswissprot'])
ensembl_gene_uniprot = ensembl_gene_uniprot[~ensembl_gene_uniprot["UniProtKB/Swiss-Prot ID"].isnull()]
ensembl_gene_uniprot.columns = ['ensembl_code', 'tar_ncRNA_code']
ensembl_gene_uniprot['rnacentral_idcode'] = [np.nan] * len(ensembl_gene_uniprot)
ensembl_gene_uniprot

,ensembl_code,tar_ncRNA_code,rnacentral_idcode
0,ENSG00000198888,P03886,NaN
1,ENSG00000198763,P03891,NaN
2,ENSG00000198804,P00395,NaN
3,ENSG00000198712,P00403,NaN
4,ENSG00000228253,P03928,NaN
...,...,...,...
38143,ENSG00000142961,Q70IA8,NaN
38144,ENSG00000116754,Q05519,NaN
38146,ENSG00000283324,A0A1B0GV90,NaN
38148,ENSG00000077585,O60478,NaN


In [122]:
mapping_file_rna_central_ensembl = "/dh-projects/uk_bb_intergenics/analysis/data/genopheno/scrnaseq/databases/RNAcentral/ensembl.tsv"
df_mapping_file_rna_central_ensembl = pd.read_csv(mapping_file_rna_central_ensembl, sep="\t", dtype=str, header= None)
df_mapping_file_rna_central_ensembl.columns = ["rnacentral_idcode", 1, 2, "species", 4, "ensembl_code"]
df_mapping_file_rna_central_ensembl = df_mapping_file_rna_central_ensembl[df_mapping_file_rna_central_ensembl["species"] == "9606"] 
df_mapping_file_rna_central_ensembl = df_mapping_file_rna_central_ensembl.loc[:, ["rnacentral_idcode", "ensembl_code"]]
df_mapping_file_rna_central_ensembl["ensembl_code"] = df_mapping_file_rna_central_ensembl["ensembl_code"].map(
    lambda ensembl_code: ensembl_code.split(".")[0])
df_mapping_file_rna_central_ensembl

,rnacentral_idcode,ensembl_code
0,URS0000000055,ENSG00000226803
1,URS00000000C9,ENSG00000248309
2,URS00000000FD,ENSG00000234279
3,URS0000000344,ENSG00000282594
4,URS0000000351,ENSG00000235427
...,...,...
1427803,URS0002186B52,ENSG00000228147
1428829,URS0002186C00,ENSG00000188004
1429187,URS0002186D5F,ENSG00000188185
1429446,URS0002186E53,ENSG00000197568


#### Other ncRNA identifiers to RNACentral 
Source: http://ftp.ebi.ac.uk/pub/databases/RNAcentral/current_release/id_mapping/database_mappings/

In [123]:
mapping_file_noncode_rna_central = "/dh-projects/uk_bb_intergenics/analysis/data/genopheno/scrnaseq/databases/RNAcentral/noncode.tsv"
df_mapping_file_noncode_rna_central = pd.read_csv(mapping_file_noncode_rna_central, sep="\t", dtype=str, header= None)
df_mapping_file_noncode_rna_central.columns = ["rnacentral_idcode", 1, 2, "species", 4, "tar_ncRNA_code"]
df_mapping_file_noncode_rna_central = df_mapping_file_noncode_rna_central[df_mapping_file_noncode_rna_central["species"] == "9606"] 
df_mapping_file_noncode_rna_central = df_mapping_file_noncode_rna_central.loc[:, ["rnacentral_idcode", "tar_ncRNA_code"]]
df_mapping_file_noncode_rna_central["tar_ncRNA_code"] = df_mapping_file_noncode_rna_central["tar_ncRNA_code"].map(
    lambda noncode_code: noncode_code.split(".")[0])

mapping_file_mirbase_rna_central = "/dh-projects/uk_bb_intergenics/analysis/data/genopheno/scrnaseq/databases/RNAcentral/mirbase.tsv"
df_mapping_file_mirbase_rna_central = pd.read_csv(mapping_file_mirbase_rna_central, sep="\t", dtype=str, header = None)
df_mapping_file_mirbase_rna_central.columns = ["rnacentral_idcode", 1, "tar_ncRNA_code", "species", 4, 5]
df_mapping_file_mirbase_rna_central = df_mapping_file_mirbase_rna_central[df_mapping_file_mirbase_rna_central["species"] == "9606"] 
df_mapping_file_mirbase_rna_central = df_mapping_file_mirbase_rna_central.loc[:, ["rnacentral_idcode", "tar_ncRNA_code"]]

In [124]:
df_mapping_file_noncode_rna_central

,rnacentral_idcode,tar_ncRNA_code
0,URS0000000055,NONHSAG044006
1,URS00000000C9,NONHSAG040980
2,URS00000000FD,NONHSAG030952
3,URS00000002D9,NONHSAG053806
4,URS0000000344,NONHSAG051423
...,...,...
171524,URS00009C5F69,NONHSAG018506
171533,URS00009C5F72,NONHSAG031885
171539,URS00009C5F78,NONHSAG031385
171543,URS00009C5F7C,NONHSAG023181


In [125]:
df_mapping_file_mirbase_rna_central

,rnacentral_idcode,tar_ncRNA_code
4,URS00000011DF,MIMAT0022711
9,URS0000001A7A,MI0003639
11,URS0000001EB3,MI0000089
125,URS00000037E8,MIMAT0019825
136,URS0000004CA3,MIMAT0018944
...,...,...
86126,URS0000D5676C,MI0039727
86598,URS0000D56919,MI0033421
87125,URS0000D56AF1,MIMAT0044658
87156,URS0000D56B0E,MIMAT0048635


In [126]:
df_mapping_file_ncRNA_rna_central = pd.concat([df_mapping_file_noncode_rna_central, df_mapping_file_mirbase_rna_central], axis=0)
df_mapping_file_ncRNA_rna_central

,rnacentral_idcode,tar_ncRNA_code
0,URS0000000055,NONHSAG044006
1,URS00000000C9,NONHSAG040980
2,URS00000000FD,NONHSAG030952
3,URS00000002D9,NONHSAG053806
4,URS0000000344,NONHSAG051423
...,...,...
86126,URS0000D5676C,MI0039727
86598,URS0000D56919,MI0033421
87125,URS0000D56AF1,MIMAT0044658
87156,URS0000D56B0E,MIMAT0048635


### Map to Ensembl Code and combine with RNACentral <-> ENSG

In [127]:
df_mapping_file_ncRNA_rna_central_ensembl = df_mapping_file_ncRNA_rna_central.join(df_mapping_file_rna_central_ensembl.set_index("rnacentral_idcode"), on="rnacentral_idcode")
df_mapping_file_ncRNA_rna_central_ensembl = df_mapping_file_ncRNA_rna_central_ensembl[~df_mapping_file_ncRNA_rna_central_ensembl["ensembl_code"].isnull()]
df_mapping_file_rna_central_ensembl["tar_ncRNA_code"] = df_mapping_file_rna_central_ensembl["ensembl_code"]
df_mapping_file_ncRNA_rna_central_ensembl = pd.concat([df_mapping_file_ncRNA_rna_central_ensembl, df_mapping_file_rna_central_ensembl, ensembl_gene_uniprot], axis=0)
df_mapping_file_ncRNA_rna_central_ensembl

,rnacentral_idcode,tar_ncRNA_code,ensembl_code
0,URS0000000055,NONHSAG044006,ENSG00000226803
1,URS00000000C9,NONHSAG040980,ENSG00000248309
2,URS00000000FD,NONHSAG030952,ENSG00000234279
4,URS0000000344,NONHSAG051423,ENSG00000282594
5,URS0000000351,NONHSAG048609,ENSG00000235427
...,...,...,...
38143,NaN,Q70IA8,ENSG00000142961
38144,NaN,Q05519,ENSG00000116754
38146,NaN,A0A1B0GV90,ENSG00000283324
38148,NaN,O60478,ENSG00000077585


### Combine npinter file with ENSG mappings

In [128]:
npinterv4 = npinterv4.join(df_mapping_file_ncRNA_rna_central_ensembl.set_index("tar_ncRNA_code"), on="ncID")
npinterv4 = npinterv4[~npinterv4["ensembl_code"].isnull()]
npinterv4.rename(columns={'rnacentral_idcode':'ncID_rnacentral_code', 'ensembl_code':'ncID_ensembl_code'}, inplace=True)
npinterv4

,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,reference,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code
3,ncRI-40000004,7SK,NONHSAG040596,lncRNA,MYC,NM_002467,protein,Electrophoretic mobility shift experiment indi...,EMSA,9018369,Homo sapiens,-,regulatory;promoter as action site,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
4,ncRI-40000005,7SK,NONHSAG040596,lncRNA,ABO,NM_020469,protein,Promoters pU6 and p7SK proved to express high ...,-,22522162,Homo sapiens,-,regulatory,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,15201869;15994294;17671421;20675720;20926576;1...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
6,ncRI-40000007,7SK,NONHSAG040596,lncRNA,Oct1,P20263,protein,PTF and Oct-1 enhance transcription from the 7...,-,1535687,Homo sapiens,-,ncRNA is regulated,regulatory,DNA-TF,Literature mining,URS00004E902D,ENSG00000249352
7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,20976203;20471949,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100531,ncRI-41100533,hsa-miR-151b,MI0003772,miRNA,NUFIP2,ENSG00000108256,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100532,ncRI-41100534,hsa-miR-151b,MI0003772,miRNA,SLC15A4,ENSG00000139370,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100533,ncRI-41100535,hsa-miR-151b,MI0003772,miRNA,TBC1D10B,ENSG00000169221,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100534,ncRI-41100536,hsa-miR-548d-5p,MI0003671,miRNA,HDLBP,ENSG00000115677,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000759C79,ENSG00000263690


### ncID GO Retrievement

In [129]:
npinterv4_ncid = npinterv4.copy()
npinterv4_ncid

,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,reference,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code
3,ncRI-40000004,7SK,NONHSAG040596,lncRNA,MYC,NM_002467,protein,Electrophoretic mobility shift experiment indi...,EMSA,9018369,Homo sapiens,-,regulatory;promoter as action site,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
4,ncRI-40000005,7SK,NONHSAG040596,lncRNA,ABO,NM_020469,protein,Promoters pU6 and p7SK proved to express high ...,-,22522162,Homo sapiens,-,regulatory,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,15201869;15994294;17671421;20675720;20926576;1...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
6,ncRI-40000007,7SK,NONHSAG040596,lncRNA,Oct1,P20263,protein,PTF and Oct-1 enhance transcription from the 7...,-,1535687,Homo sapiens,-,ncRNA is regulated,regulatory,DNA-TF,Literature mining,URS00004E902D,ENSG00000249352
7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,20976203;20471949,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100531,ncRI-41100533,hsa-miR-151b,MI0003772,miRNA,NUFIP2,ENSG00000108256,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100532,ncRI-41100534,hsa-miR-151b,MI0003772,miRNA,SLC15A4,ENSG00000139370,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100533,ncRI-41100535,hsa-miR-151b,MI0003772,miRNA,TBC1D10B,ENSG00000169221,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100534,ncRI-41100536,hsa-miR-548d-5p,MI0003671,miRNA,HDLBP,ENSG00000115677,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000759C79,ENSG00000263690


In [130]:
npinterv4_ncid["ncID"] = npinterv4_ncid[['interID', 'ncName', 'ncID', 'ncType', 'tarName', 'tarID', 'tarType', 'interDescription', 'experiment', 'reference', 'organism', 'tissueOrCell', 'tag', 'class', 'level', 'datasource', 'ncID_rnacentral_code', 'ncID_ensembl_code']].groupby(['ncID_ensembl_code'])["ncID"].transform(lambda x: '; '.join(x))
npinterv4_ncid["ncType"] = npinterv4_ncid[['interID', 'ncName', 'ncID', 'ncType', 'tarName', 'tarID', 'tarType', 'interDescription', 'experiment', 'reference', 'organism', 'tissueOrCell', 'tag', 'class', 'level', 'datasource', 'ncID_rnacentral_code', 'ncID_ensembl_code']].groupby(['ncID_ensembl_code'])["ncType"].transform(lambda x: '; '.join(x))
npinterv4_ncid["reference"] = npinterv4_ncid[['interID', 'ncName', 'ncID', 'ncType', 'tarName', 'tarID', 'tarType', 'interDescription', 'experiment', 'reference', 'organism', 'tissueOrCell', 'tag', 'class', 'level', 'datasource', 'ncID_rnacentral_code', 'ncID_ensembl_code']].groupby(['ncID_ensembl_code'])["reference"].transform(lambda x: '; '.join(x))
npinterv4_ncid = npinterv4_ncid[["ncID", "ncType", "reference", 'ncID_ensembl_code', 'ncID_rnacentral_code', "tag", "level"]].drop_duplicates()
npinterv4_ncid["tag"] = npinterv4_ncid[["ncID", "ncType", "reference", "ncID_ensembl_code", "ncID_rnacentral_code", "tag", "level"]].groupby(["ncID_ensembl_code"])["tag"].transform(lambda x: ';'.join(x))
npinterv4_ncid["level"] = npinterv4_ncid[["ncID", "ncType", "reference", "ncID_ensembl_code", "ncID_rnacentral_code", "tag", "level"]].groupby(["ncID_ensembl_code"])["level"].transform(lambda x: ';'.join(x))
npinterv4_ncid = npinterv4_ncid[["ncID", "ncType", "reference", "ncID_ensembl_code", "ncID_rnacentral_code", "tag", "level"]].drop_duplicates()
npinterv4_ncid

,ncID,ncType,reference,ncID_ensembl_code,ncID_rnacentral_code,tag,level
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...
24,NONHSAG048195; NONHSAG048195; NONHSAG048195; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 24164894;22955616; 25301781; 2384659...,ENSG00000243107,URS0000539BDF,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein;RNA-RNA
25,NONHSAG048720; NONHSAG048720; NONHSAG048720; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; 24332178; -; 23023293; 263...,ENSG00000226770,URS0000477DCE,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein
28,NONHSAG048153; NONHSAG048153; NONHSAG048153; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; -; 23023293; 26321680; 280...,ENSG00000233420,URS00003A4444,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein
29,NONHSAG019101; NONHSAG019101; NONHSAG019101; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25531890; 25301781; 25301781; -; -; ...,ENSG00000235560,URS000028AB2B,miRNA target interaction;RNA-RNA interaction;m...,RNA-RNA;RNA-RNA;RNA-Protein;RNA-Protein;RNA-RN...
...,...,...,...,...,...,...,...
1100453,ENSG00000199787,snoRNA,27184080,ENSG00000199787,URS000063D36F,RNA-RNA interaction,RNA-RNA
1100455,ENSG00000263764,snoRNA,27184080,ENSG00000263764,URS000063F175,RNA-RNA interaction,RNA-RNA
1100456,ENSG00000208892,snoRNA,27184080,ENSG00000208892,URS00006276F4,RNA-RNA interaction,RNA-RNA
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA


In [131]:
npinterv4_ncid["go_terms"] = np.nan
npinterv4_ncid["taxonomy"] = 9606
npinterv4_ncid

,ncID,ncType,reference,ncID_ensembl_code,ncID_rnacentral_code,tag,level,go_terms,taxonomy
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,NaN,9606
24,NONHSAG048195; NONHSAG048195; NONHSAG048195; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 24164894;22955616; 25301781; 2384659...,ENSG00000243107,URS0000539BDF,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein;RNA-RNA,NaN,9606
25,NONHSAG048720; NONHSAG048720; NONHSAG048720; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; 24332178; -; 23023293; 263...,ENSG00000226770,URS0000477DCE,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein,NaN,9606
28,NONHSAG048153; NONHSAG048153; NONHSAG048153; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; -; 23023293; 26321680; 280...,ENSG00000233420,URS00003A4444,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein,NaN,9606
29,NONHSAG019101; NONHSAG019101; NONHSAG019101; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25531890; 25301781; 25301781; -; -; ...,ENSG00000235560,URS000028AB2B,miRNA target interaction;RNA-RNA interaction;m...,RNA-RNA;RNA-RNA;RNA-Protein;RNA-Protein;RNA-RN...,NaN,9606
...,...,...,...,...,...,...,...,...,...
1100453,ENSG00000199787,snoRNA,27184080,ENSG00000199787,URS000063D36F,RNA-RNA interaction,RNA-RNA,NaN,9606
1100455,ENSG00000263764,snoRNA,27184080,ENSG00000263764,URS000063F175,RNA-RNA interaction,RNA-RNA,NaN,9606
1100456,ENSG00000208892,snoRNA,27184080,ENSG00000208892,URS00006276F4,RNA-RNA interaction,RNA-RNA,NaN,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,NaN,9606


#### NC-iD GO-Terms Mapping

In [132]:
#For the nc version the GO terms are assigned to specific genes by specific tags and keywords in the NPInter database.
#The exact dictionary explaining the assignment of GO terms based on these tags is given here.
# For a simplified version please also refer to ./resources/single_cell/mapping_keyword_go_term_npinterv4.json.

manually_mapped_go_terms = {
    "bp_only": {
        "promoter as action site": "GO:0010468", #regulation of gene expression
        "ncRNA affects synthesis or function of protein": "GO:0040033",
        "expression, or processing, or function of ncRNA is affected": "GO:0034661", #ncRNA catabolic process
        "ncRNA targets mRNA": "GO:0016071"
    },
    "bp_mf": {
        "miRNA target interaction": "GO:0010586;GO:0010467;GO:0035198", #miRNA metabolic process
        "miRNA": "GO:0010586;GO:0035198", #miRNA metabolic process
        "promoter as action site": "GO:0001046",
        "DNA-TF": "GO:0008134;GO:0005515",
        "RNA-TF": "GO:0008134;GO:0005515",
        "lncRNA-TF": "GO:0008134;GO:0005515",
        "RNA-DNA": "GO:0003677;GO:0003723",
        "RNA-RNA": "GO:0003723",
        "DNA-DNA": "GO:0003677",
        "RNA-Protein": "GO:0005515",
        "DNA-Protein": "GO:0005515"
    }
}

#####################################################################################################


npinterv4_ncid_go_terms = npinterv4_ncid.copy()

for index, npinterv4_ncid_go_terms_uniprot_row in npinterv4_ncid_go_terms.iterrows():
    
    list_tags = set(npinterv4_ncid_go_terms_uniprot_row["tag"].split(";"))

    list_level = set(npinterv4_ncid_go_terms_uniprot_row["level"].split(";"))

    list_go_criteria = list_tags | list_level
    
    mf_false = True

    go_terms = set()

    for go_criteria in list_go_criteria:
        go = manually_mapped_go_terms["bp_only"].get(go_criteria)
        if go is not None:
            go_terms.add(go)
        go =  manually_mapped_go_terms["bp_mf"].get(go_criteria)
        if go is not None:
            go_terms.add(go)
            mf_false = False      

    go_terms.add("GO:0034660")

    if mf_false == True:
        go_terms.add("GO:0003674")
    
    go_terms = ";".join(go_terms)

    npinterv4_ncid_go_terms.loc[index, "go_terms"] = go_terms
    
npinterv4_ncid_go_terms

,ncID,ncType,reference,ncID_ensembl_code,ncID_rnacentral_code,tag,level,go_terms,taxonomy
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0010468;GO:0040033;GO:0003723;GO:0034660;GO...,9606
24,NONHSAG048195; NONHSAG048195; NONHSAG048195; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 24164894;22955616; 25301781; 2384659...,ENSG00000243107,URS0000539BDF,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein;RNA-RNA,GO:0010586;GO:0035198;GO:0003723;GO:0034660;GO...,9606
25,NONHSAG048720; NONHSAG048720; NONHSAG048720; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; 24332178; -; 23023293; 263...,ENSG00000226770,URS0000477DCE,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein,GO:0010586;GO:0010467;GO:0035198;GO:0003723;GO...,9606
28,NONHSAG048153; NONHSAG048153; NONHSAG048153; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25301781; -; 23023293; 26321680; 280...,ENSG00000233420,URS00003A4444,miRNA target interaction;RNA-RNA interaction;n...,RNA-RNA;RNA-Protein,GO:0010586;GO:0010467;GO:0035198;GO:0003723;GO...,9606
29,NONHSAG019101; NONHSAG019101; NONHSAG019101; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,25531890; 25531890; 25301781; 25301781; -; -; ...,ENSG00000235560,URS000028AB2B,miRNA target interaction;RNA-RNA interaction;m...,RNA-RNA;RNA-RNA;RNA-Protein;RNA-Protein;RNA-RN...,GO:0010586;GO:0035198;GO:0003723;GO:0034660;GO...,9606
...,...,...,...,...,...,...,...,...,...
1100453,ENSG00000199787,snoRNA,27184080,ENSG00000199787,URS000063D36F,RNA-RNA interaction,RNA-RNA,GO:0003723;GO:0034660,9606
1100455,ENSG00000263764,snoRNA,27184080,ENSG00000263764,URS000063F175,RNA-RNA interaction,RNA-RNA,GO:0003723;GO:0034660,9606
1100456,ENSG00000208892,snoRNA,27184080,ENSG00000208892,URS00006276F4,RNA-RNA interaction,RNA-RNA,GO:0003723;GO:0034660,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,GO:0003723;GO:0034660,9606


In [133]:
npinterv4_ncid_go_terms["go_terms"] = npinterv4_ncid_go_terms["go_terms"].apply(lambda x: x.split(";"))
npinterv4_ncid_go_terms = npinterv4_ncid_go_terms.explode('go_terms')
npinterv4_ncid_go_terms

,ncID,ncType,reference,ncID_ensembl_code,ncID_rnacentral_code,tag,level,go_terms,taxonomy
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0010468,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0040033,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0003723,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0034660,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0005515,9606
...,...,...,...,...,...,...,...,...,...
1100456,ENSG00000208892,snoRNA,27184080,ENSG00000208892,URS00006276F4,RNA-RNA interaction,RNA-RNA,GO:0034660,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,GO:0003723,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,GO:0034660,9606
1100515,MI0019145; MI0019145,miRNA; miRNA,23622248; 23622248,ENSG00000283146,URS000075B121,RNA-RNA interaction,RNA-RNA,GO:0003723,9606


In [134]:
### Drop all rows, which already are annotated

In [135]:
index_list_coocurrence_nc = np.in1d(np.array(npinterv4_ncid_go_terms["ncID_ensembl_code"]), list(set(npinterv4_ncid_go_terms["ncID_ensembl_code"]) & set(np.unique(gaf[1])))).nonzero()[0]
index_list_coocurrence_nc

array([], dtype=int64)

In [136]:
len(index_list_coocurrence_nc)

0

In [137]:
npinterv4_ncid_go_terms.drop(index_list_coocurrence_nc, inplace=True)
npinterv4_ncid_go_terms

,ncID,ncType,reference,ncID_ensembl_code,ncID_rnacentral_code,tag,level,go_terms,taxonomy
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0010468,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0040033,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0003723,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0034660,9606
3,NONHSAG040596; NONHSAG040596; NONHSAG040596; N...,lncRNA; lncRNA; lncRNA; lncRNA; lncRNA; lncRNA...,9018369; 22522162; 15201869;15994294;17671421;...,ENSG00000249352,URS00004E902D,regulatory;promoter as action site;regulatory;...,RNA-RNA;RNA-RNA;RNA-Protein;DNA-TF;RNA-Protein...,GO:0005515,9606
...,...,...,...,...,...,...,...,...,...
1100456,ENSG00000208892,snoRNA,27184080,ENSG00000208892,URS00006276F4,RNA-RNA interaction,RNA-RNA,GO:0034660,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,GO:0003723,9606
1100460,ENSG00000200753,snoRNA,27184080,ENSG00000200753,URS000006D2A0,RNA-RNA interaction,RNA-RNA,GO:0034660,9606
1100515,MI0019145; MI0019145,miRNA; miRNA,23622248; 23622248,ENSG00000283146,URS000075B121,RNA-RNA interaction,RNA-RNA,GO:0003723,9606


#### Save file

In [138]:
npinterv4_ncid_go_terms_df = pd.DataFrame({
    0: ["NPInter"] * len(npinterv4_ncid_go_terms),
    1: npinterv4_ncid_go_terms["ncID_ensembl_code"],
    2: list(npinterv4_ncid_go_terms.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="ncID_ensembl_code")["Gene name"].astype("str")),
    3: ["enables"]  * len(npinterv4_ncid_go_terms),
    4: npinterv4_ncid_go_terms["go_terms"],
    5: ["GO_REF:0000001"] * len(npinterv4_ncid_go_terms),
    6: ["EXP"] * len(npinterv4_ncid_go_terms),
    7: ["NPInter"] * len(npinterv4_ncid_go_terms),
    8: ["P" if obodag[go_term].namespace == "biological_process" else "F" if obodag[go_term].namespace == "molecular_function" else "C" for go_term in npinterv4_ncid_go_terms["go_terms"]], 
    9: [obodag[go_term].name for go_term in npinterv4_ncid_go_terms["go_terms"]],
    10: list(npinterv4_ncid_go_terms.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="ncID_ensembl_code")["Gene name"].astype("str")),
    11: ["ncRNA"] * len(npinterv4_ncid_go_terms),
    12: ["taxon:9606"] * len(npinterv4_ncid_go_terms),
    13: [20220215] * len(npinterv4_ncid_go_terms),
    14: ["NPInter"] * len(npinterv4_ncid_go_terms),
    15: [np.nan] * len(npinterv4_ncid_go_terms),
    16: [np.nan] * len(npinterv4_ncid_go_terms)
})

npinterv4_ncid_go_terms_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0010468,GO_REF:0000001,EXP,NPInter,P,regulation of gene expression,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0040033,GO_REF:0000001,EXP,NPInter,P,"negative regulation of translation, ncRNA-medi...",LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0005515,GO_REF:0000001,EXP,NPInter,F,protein binding,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100456,NPInter,ENSG00000208892,SNORA49,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,SNORA49,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENSG00000200753,nan,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENSG00000200753,nan,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100515,NPInter,ENSG00000283146,MIR548AU,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,MIR548AU,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [139]:
from itertools import islice
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human.gaf', "r") as myfile:
    head = list(islice(myfile, 41))
head

['!gaf-version: 2.2\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T16:45\n',
 '!\n',
 '!Header from source association file:\n',
 '!=================================\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T04:41\n',
 '!\n',
 '!Header from goa_human source association file:\n',
 '!=================================\n',
 '!\n',
 '!The set of protein accessions included in this file is based on UniProt reference proteomes, which provide one protein per gene.\n',
 '!They include the protein sequences annotated in Swiss-Prot or the longest TrEMBL transcript if there is no Swiss-Prot record.\n',
 '!If a particular protein accession is not annotated with GO, then it will not appear in this file.\n',
 '!\n',
 '!Note that the annotation set in this file is filtered in order to reduce redundancy; the full, unfiltered set can be found in\n',
 '!ftp://ftp.ebi.ac.uk/pub/databases/GO/goa/UNIPROT/goa_uniprot_all.gz\n',
 '!\n',
 '!date-gene

In [140]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_nc_ensembl_gene_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [141]:
npinterv4_ncid_go_terms_df.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_nc_ensembl_gene_mapping.gaf', mode="a", sep="\t", header=False, index=False)

### tarID GO Retrievement

In [23]:
npinterv4_tarid = npinterv4.copy()
npinterv4_tarid

,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,reference,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code
3,ncRI-40000004,7SK,NONHSAG040596,lncRNA,MYC,NM_002467,protein,Electrophoretic mobility shift experiment indi...,EMSA,9018369,Homo sapiens,-,regulatory;promoter as action site,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
4,ncRI-40000005,7SK,NONHSAG040596,lncRNA,ABO,NM_020469,protein,Promoters pU6 and p7SK proved to express high ...,-,22522162,Homo sapiens,-,regulatory,regulatory,RNA-RNA,Literature mining,URS00004E902D,ENSG00000249352
5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,15201869;15994294;17671421;20675720;20926576;1...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
6,ncRI-40000007,7SK,NONHSAG040596,lncRNA,Oct1,P20263,protein,PTF and Oct-1 enhance transcription from the 7...,-,1535687,Homo sapiens,-,ncRNA is regulated,regulatory,DNA-TF,Literature mining,URS00004E902D,ENSG00000249352
7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,20976203;20471949,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100531,ncRI-41100533,hsa-miR-151b,MI0003772,miRNA,NUFIP2,ENSG00000108256,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100532,ncRI-41100534,hsa-miR-151b,MI0003772,miRNA,SLC15A4,ENSG00000139370,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100533,ncRI-41100535,hsa-miR-151b,MI0003772,miRNA,TBC1D10B,ENSG00000169221,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000075C169,ENSG00000265154
1100534,ncRI-41100536,hsa-miR-548d-5p,MI0003671,miRNA,HDLBP,ENSG00000115677,mRNA,RNA-RNA interaction from RISE database,CLASH,23622248,Homo sapiens,HEK293,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000759C79,ENSG00000263690


In [24]:
npinterv4_tarid = npinterv4_tarid.join(df_mapping_file_ncRNA_rna_central_ensembl.set_index("tar_ncRNA_code"), on="tarID")
npinterv4_tarid = npinterv4_tarid[~npinterv4_tarid["ensembl_code"].isnull()]
npinterv4_tarid.rename(columns={'rnacentral_idcode':'tarID_rnacentral_code', 'ensembl_code':'tarID_ensembl_code'}, inplace=True)
npinterv4_tarid = npinterv4_tarid.drop_duplicates()
npinterv4_tarid = npinterv4_tarid.reset_index()
npinterv4_tarid

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834
1,7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000198650
2,8,ncRI-40000009,7SK,NONHSAG040596,lncRNA,CDK9,P50750,protein,Northern hybridization was performed to confir...,Northern Blot,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000136807
3,9,ncRI-40000010,7SK,NONHSAG040596,lncRNA,PPP1CA,P62136,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000172531
4,10,ncRI-40000011,7SK,NONHSAG040596,lncRNA,PP2B,P62140,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000213639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487273,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6AE6,ENSG00000255717
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717
487275,1100458,ncRI-41100460,SNORD117,ENSG00000266295,snoRNA,SNORA5C,ENSG00000201772,snoRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS00002F13BD,ENSG00000266295,URS000072157F,ENSG00000201772
487276,1100459,ncRI-41100461,U6,ENSG00000278140,snoRNA,SNORA79,ENSG00000222489,snRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000671B81,ENSG00000278140,URS0000704142,ENSG00000222489


In [25]:
len(set(npinterv4_tarid["tarID_ensembl_code"]))

2068

In [26]:
len(set(npinterv4_tarid["ncID_ensembl_code"]) & set(npinterv4_tarid["tarID_ensembl_code"]))

594

#### Tar-ID GO-Terms Mapping

In [27]:
npinterv4_tarid_go_terms = npinterv4_tarid.copy()
npinterv4_tarid_go_terms

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834
1,7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000198650
2,8,ncRI-40000009,7SK,NONHSAG040596,lncRNA,CDK9,P50750,protein,Northern hybridization was performed to confir...,Northern Blot,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000136807
3,9,ncRI-40000010,7SK,NONHSAG040596,lncRNA,PPP1CA,P62136,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000172531
4,10,ncRI-40000011,7SK,NONHSAG040596,lncRNA,PP2B,P62140,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000213639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487273,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6AE6,ENSG00000255717
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717
487275,1100458,ncRI-41100460,SNORD117,ENSG00000266295,snoRNA,SNORA5C,ENSG00000201772,snoRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS00002F13BD,ENSG00000266295,URS000072157F,ENSG00000201772
487276,1100459,ncRI-41100461,U6,ENSG00000278140,snoRNA,SNORA79,ENSG00000222489,snRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000671B81,ENSG00000278140,URS0000704142,ENSG00000222489


In [28]:
len(np.unique(npinterv4_tarid_go_terms["tarID_ensembl_code"]))

2068

#### V1 - Target of target is target - VERY SLOW

In [65]:
#V1 Target of target is target
go_terms_npinterv4_tarid_go_terms = {}

for unique_tarid in tqdm(np.unique(npinterv4_tarid_go_terms["tarID_ensembl_code"])):
        
    tarID_ensembl_codes = set()
    tarID_ensembl_codes.add(unique_tarid)
    ncRNA_tar_interactions_old = set([unique_tarid])
    further_ncRNA_tar_interaction = True
    go_terms = set()
    
    print(unique_tarid)

    while further_ncRNA_tar_interaction:
        ncRNA_tar_interactions = set()
        for ncRNA_tar_interaction_old in ncRNA_tar_interactions_old:
            ncRNA_tar_interactions =  ncRNA_tar_interactions | set(np.unique(npinterv4_tarid_go_terms[npinterv4_tarid_go_terms["ncID_ensembl_code"] == ncRNA_tar_interaction_old]["tarID_ensembl_code"]))
        if len(ncRNA_tar_interactions - tarID_ensembl_codes) >= 1:
            print("hjsdghjag")
            ncRNA_tar_interactions_old = ncRNA_tar_interactions 
            tarID_ensembl_codes = tarID_ensembl_codes | set(ncRNA_tar_interactions)
        else:
            further_ncRNA_tar_interaction = False
            
    for tarID_ensembl_code in tarID_ensembl_codes:
        go_terms = go_terms | set(gaf[gaf[1] == tarID_ensembl_code][4])
    if len(go_terms) > 0:
        go_terms_npinterv4_tarid_go_terms[unique_tarid] = list(go_terms)
    else:
        go_terms_npinterv4_tarid_go_terms[unique_tarid] = np.nan

  0%|          | 2/2068 [00:00<02:21, 14.60it/s]

ENSG00000001631
ENSG00000003756
ENSG00000004478
ENSG00000004487


  0%|          | 8/2068 [00:00<02:09, 15.85it/s]

ENSG00000004534
ENSG00000004700
ENSG00000005007
ENSG00000005421
ENSG00000006712


  1%|          | 12/2068 [00:00<02:08, 16.04it/s]

...

100%|█████████▉| 2065/2068 [21:38:22<02:40, 53.46s/it]

hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag


100%|█████████▉| 2067/2068 [21:42:31<01:18, 78.69s/it]

ENSG00000288462
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag
hjsdghjag


100%|██████████| 2068/2068 [21:47:27<00:00, 37.93s/it] 


import pickle

with open('/home/arnoldtl/go_terms_npinterv4_tarid_go_terms_safety_copy.pickle', 'wb') as handle:
    pickle.dump(go_terms_npinterv4_tarid_go_terms, handle, protocol=pickle.DEFAULT_PROTOCOL)

In [70]:
go_terms_npinterv4_tarid_go_terms

{'ENSG00000001631': ['GO:0005856',
  'GO:0050790',
  'GO:0005911',
  'GO:0005546',
  'GO:0005737',
  'GO:0008017',
  'GO:0005515',
  'GO:0045454',
  'GO:0005615',
  'GO:0005886',
  'GO:0030695',
  'GO:0001525',
  'GO:2000352',
  'GO:0016525',
  'GO:2000114',
  'GO:0007264',
  'GO:0001937',
  'GO:0010596',
  'GO:0033622'],
 'ENSG00000003756': ['GO:0006396',
  'GO:0000398',
  'GO:0005654',
  'GO:0003723',
  'GO:0005634',
  'GO:0046872',
  'GO:0003729',
  'GO:0003677',
  'GO:0005681',
  'GO:0043065',
  'GO:0005515',
  'GO:0006915',
  'GO:0000381',
  'GO:0000245',
  'GO:0008285'],
 'ENSG00000004478': ['GO:0005739',
  'GO:0070062',
  'GO:0061077',
  'GO:0048156',
  'GO:0031111',
  'GO:0003723',
  'GO:0031503',
  'GO:0006457',
  'GO:0005525',
  'GO:0032991',
  'GO:0031115',
  'GO:0005737',
  'GO:0005515',
  'GO:0030850',
  'GO:0005528',
  'GO:0003755',
  'GO:0007566',
  'GO:0044295',
  'GO:0035259',
  'GO:0010977',
  'GO:0032767',
  'GO:0005524',
  'GO:0043025',
  'GO:0005654',
  'GO:0006825

In [75]:
go_terms_npinterv4_tarid_go_terms_df = pd.DataFrame({"ensg_npintertar": list(go_terms_npinterv4_tarid_go_terms.keys()), "go_terms": list(go_terms_npinterv4_tarid_go_terms.values())})
go_terms_npinterv4_tarid_go_terms_df

,ensg_npintertar,go_terms
0,ENSG00000001631,"[GO:0005856, GO:0050790, GO:0005911, GO:000554..."
1,ENSG00000003756,"[GO:0006396, GO:0000398, GO:0005654, GO:000372..."
2,ENSG00000004478,"[GO:0005739, GO:0070062, GO:0061077, GO:004815..."
3,ENSG00000004487,"[GO:0043426, GO:0035563, GO:0045793, GO:200064..."
4,ENSG00000004534,"[GO:0006396, GO:0000398, GO:0005515, GO:000372..."
...,...,...
2063,ENSG00000288329,NaN
2064,ENSG00000288358,NaN
2065,ENSG00000288367,"[GO:0030154, GO:0000785, GO:0038183, GO:004589..."
2066,ENSG00000288397,"[GO:0061630, GO:0043398, GO:0048333, GO:014007..."


In [76]:
npinterv4_tarid_go_terms = npinterv4_tarid_go_terms.join(go_terms_npinterv4_tarid_go_terms_df.set_index("ensg_npintertar"), on="tarID_ensembl_code")
npinterv4_tarid_go_terms = npinterv4_tarid_go_terms[~npinterv4_tarid_go_terms["go_terms"].isnull()]
npinterv4_tarid_go_terms

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,"[GO:0106140, GO:1901798, GO:0004860, GO:004280..."
1,7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000198650,"[GO:0005739, GO:0006559, GO:0042802, GO:000582..."
2,8,ncRI-40000009,7SK,NONHSAG040596,lncRNA,CDK9,P50750,protein,Northern hybridization was performed to confir...,Northern Blot,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000136807,"[GO:0004712, GO:0051726, GO:0004672, GO:190365..."
3,9,ncRI-40000010,7SK,NONHSAG040596,lncRNA,PPP1CA,P62136,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000172531,"[GO:0070062, GO:0032922, GO:0004721, GO:009864..."
4,10,ncRI-40000011,7SK,NONHSAG040596,lncRNA,PP2B,P62140,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000213639,"[GO:0070062, GO:0032922, GO:0019901, GO:000565..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487270,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF3F1E,ENSG00000255717,"[GO:0061630, GO:0043398, GO:0048333, GO:014007..."
487271,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF4F8E,ENSG00000255717,"[GO:0061630, GO:0043398, GO:0048333, GO:014007..."
487272,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF5E17,ENSG00000255717,"[GO:0061630, GO:0043398, GO:0048333, GO:014007..."
487273,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6AE6,ENSG00000255717,"[GO:0061630, GO:0043398, GO:0048333, GO:014007..."


In [77]:
npinterv4_tarid_go_terms = npinterv4_tarid_go_terms.explode('go_terms')
npinterv4_tarid_go_terms

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0106140
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:1901798
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0004860
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0042802
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0007507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035522
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035455
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0006995
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0005852


In [81]:
npinterv4_tarid_go_terms = npinterv4_tarid_go_terms[npinterv4_tarid_go_terms["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0106140
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:1901798
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0004860
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0042802
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0007507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035522
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035455
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0006995
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0005852


In [ ]:
for go_term in np.unique(npinterv4_tarid_go_terms["go_terms"]):
    try:
        obodag[go_term]
    except KeyError:
        print(go_term)

In [ ]:
print("ksdhkg")

##### Save file

In [ ]:
npinterv4_tarid_go_terms_df = pd.DataFrame({
    0: ["NPInter"] * len(npinterv4_tarid_go_terms),
    1: npinterv4_tarid_go_terms["tarID_ensembl_code"],
    2: list(npinterv4_tarid_go_terms.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    3: ["enables"]  * len(npinterv4_tarid_go_terms),
    4: npinterv4_tarid_go_terms["go_terms"],
    5: ["NPInter"] * len(npinterv4_tarid_go_terms),
    6: ["EXP"] * len(npinterv4_tarid_go_terms),
    7: ["NPInter"] * len(npinterv4_tarid_go_terms),
    8: [np.nan if go_term not in list(obodag.keys()) else "P" if obodag[go_term].namespace == "biological_process" else "F" if obodag[go_term].namespace == "molecular_function" else "C" for go_term in npinterv4_tarid_go_terms_v2["go_terms"]], 
    9: [np.nan if go_term not in list(obodag.keys()) else obodag[go_term].name for go_term in npinterv4_tarid_go_terms["go_terms"]],
    10: list(npinterv4_tarid_go_terms.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    11: ["ncRNA"] * len(npinterv4_tarid_go_terms),
    12: ["taxon:9606"] * len(npinterv4_tarid_go_terms),
    13: ["NPInter"] * len(npinterv4_tarid_go_terms),
    14: ["NPInter"] * len(npinterv4_tarid_go_terms),
    15: [np.nan] * len(npinterv4_tarid_go_terms),
    16: [np.nan] * len(npinterv4_tarid_go_terms)
})

npinterv4_tarid_go_terms_df

In [ ]:
npinterv4_tarid_go_terms_df = npinterv4_tarid_go_terms_df[~npinterv4_tarid_go_terms_df[8].isnull()]
npinterv4_tarid_go_terms_df

In [ ]:
from itertools import islice
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human.gaf', "r") as myfile:
    head = list(islice(myfile, 41))
head

In [ ]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_ensembl_gene_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [ ]:
npinterv4_tarid_go_terms_df.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_ensembl_gene_mapping.gaf', mode="a", sep="\t", header=False, index=False)

#### V2 - Only direct target

In [96]:
npinterv4_tarid_go_terms_v2 = npinterv4_tarid.copy()
npinterv4_tarid_go_terms_v2

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,organism,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834
1,7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000198650
2,8,ncRI-40000009,7SK,NONHSAG040596,lncRNA,CDK9,P50750,protein,Northern hybridization was performed to confir...,Northern Blot,...,Homo sapiens,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000136807
3,9,ncRI-40000010,7SK,NONHSAG040596,lncRNA,PPP1CA,P62136,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000172531
4,10,ncRI-40000011,7SK,NONHSAG040596,lncRNA,PP2B,P62140,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,Homo sapiens,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000213639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487273,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6AE6,ENSG00000255717
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717
487275,1100458,ncRI-41100460,SNORD117,ENSG00000266295,snoRNA,SNORA5C,ENSG00000201772,snoRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS00002F13BD,ENSG00000266295,URS000072157F,ENSG00000201772
487276,1100459,ncRI-41100461,U6,ENSG00000278140,snoRNA,SNORA79,ENSG00000222489,snRNA,RNA-RNA interaction from RISE database,LIGRseq,...,Homo sapiens,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS0000671B81,ENSG00000278140,URS0000704142,ENSG00000222489


In [93]:
go_terms_npinterv4_tarid_go_terms_v2 = {}

for unique_tarid in tqdm(np.unique(npinterv4_tarid_go_terms["tarID_ensembl_code"])):
        
    tarID_ensembl_codes = set()
    tarID_ensembl_codes.add(unique_tarid)
    #go_terms = set()
    
    print(unique_tarid)

    #tarID_ensembl_codes = set(np.unique(npinterv4_tarid_go_terms[npinterv4_tarid_go_terms["ncID_ensembl_code"] == unique_tarid]["tarID_ensembl_code"]))
    #break
    #for tarID_ensembl_code in tarID_ensembl_codes:
    go_terms = set(gaf[gaf[1] == unique_tarid][4])
    
    if len(go_terms) > 0:
        go_terms_npinterv4_tarid_go_terms_v2[unique_tarid] = list(go_terms)
    else:
        go_terms_npinterv4_tarid_go_terms_v2[unique_tarid] = np.nan

  0%|          | 6/1375 [00:00<00:49, 27.52it/s]

ENSG00000001631
ENSG00000003756
ENSG00000004478
ENSG00000004487
ENSG00000004534
ENSG00000004700
ENSG00000005007


  1%|          | 12/1375 [00:00<00:49, 27.73it/s]

ENSG00000005421
ENSG00000006712
ENSG00000006744
ENSG00000007372
ENSG00000007392
ENSG00000007908
ENSG00000011243


  1%|▏         | 18/1375 [00:00<00:50, 26.87it/s]

...

ENSG00000284536
ENSG00000284538
ENSG00000284564
ENSG00000284565
ENSG00000284567
ENSG00000284586
ENSG00000285049


 99%|█████████▉| 1368/1375 [00:50<00:00, 26.92it/s]

ENSG00000285056
ENSG00000285161
ENSG00000285317
ENSG00000285438
ENSG00000288121
ENSG00000288130
ENSG00000288145


100%|██████████| 1375/1375 [00:50<00:00, 27.04it/s]

ENSG00000288183
ENSG00000288210
ENSG00000288237
ENSG00000288367
ENSG00000288397
ENSG00000288462


In [94]:
go_terms_npinterv4_tarid_go_terms_v2

{'ENSG00000001631': ['GO:0005856',
  'GO:0005737',
  'GO:0008017',
  'GO:0005515',
  'GO:0045454',
  'GO:0005615',
  'GO:0005886',
  'GO:2000114',
  'GO:0001937',
  'GO:0010596',
  'GO:0050790',
  'GO:0005911',
  'GO:0005546',
  'GO:0030695',
  'GO:0001525',
  'GO:2000352',
  'GO:0016525',
  'GO:0007264',
  'GO:0033622'],
 'ENSG00000003756': ['GO:0006396',
  'GO:0000398',
  'GO:0005654',
  'GO:0003723',
  'GO:0005634',
  'GO:0046872',
  'GO:0003729',
  'GO:0003677',
  'GO:0005681',
  'GO:0043065',
  'GO:0005515',
  'GO:0006915',
  'GO:0000381',
  'GO:0000245',
  'GO:0008285'],
 'ENSG00000004478': ['GO:0005739',
  'GO:0070062',
  'GO:0061077',
  'GO:0048156',
  'GO:0031111',
  'GO:0003723',
  'GO:0031503',
  'GO:0006457',
  'GO:0005525',
  'GO:0032991',
  'GO:0031115',
  'GO:0005737',
  'GO:0005515',
  'GO:0030850',
  'GO:0005528',
  'GO:0003755',
  'GO:0007566',
  'GO:0044295',
  'GO:0035259',
  'GO:0010977',
  'GO:0032767',
  'GO:0005524',
  'GO:0043025',
  'GO:0005654',
  'GO:0006825

In [95]:
go_terms_npinterv4_tarid_go_terms_df_v2 = pd.DataFrame({"ensg_npintertar": list(go_terms_npinterv4_tarid_go_terms_v2.keys()), "go_terms": list(go_terms_npinterv4_tarid_go_terms_v2.values())})
go_terms_npinterv4_tarid_go_terms_df_v2

,ensg_npintertar,go_terms
0,ENSG00000001631,"[GO:0005856, GO:0005737, GO:0008017, GO:000551..."
1,ENSG00000003756,"[GO:0006396, GO:0000398, GO:0005654, GO:000372..."
2,ENSG00000004478,"[GO:0005739, GO:0070062, GO:0061077, GO:004815..."
3,ENSG00000004487,"[GO:0090308, GO:0120162, GO:0010976, GO:004342..."
4,ENSG00000004534,"[GO:0006396, GO:0000398, GO:0003723, GO:000563..."
...,...,...
1370,ENSG00000288210,"[GO:0006336, GO:0030154, GO:0000785, GO:000565..."
1371,ENSG00000288237,NaN
1372,ENSG00000288367,NaN
1373,ENSG00000288397,NaN


In [97]:
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2.join(go_terms_npinterv4_tarid_go_terms_df_v2.set_index("ensg_npintertar"), on="tarID_ensembl_code")
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[~npinterv4_tarid_go_terms_v2["go_terms"].isnull()]
npinterv4_tarid_go_terms_v2

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,"[GO:0106140, GO:0004860, GO:0042802, GO:000750..."
1,7,ncRI-40000008,7SK,NONHSAG040596,lncRNA,Tat,P17735,protein,Tat efficiently replaces HEXIM1 on the 7SK snR...,-,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000198650,"[GO:0005739, GO:0016597, GO:0006559, GO:000483..."
2,8,ncRI-40000009,7SK,NONHSAG040596,lncRNA,CDK9,P50750,protein,Northern hybridization was performed to confir...,Northern Blot,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000136807,"[GO:0004712, GO:0051726, GO:0004672, GO:190365..."
3,9,ncRI-40000010,7SK,NONHSAG040596,lncRNA,PPP1CA,P62136,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000172531,"[GO:0070062, GO:0032922, GO:0004721, GO:009864..."
4,10,ncRI-40000011,7SK,NONHSAG040596,lncRNA,PP2B,P62140,protein,PP2B and PP1alpha cooperatively disrupt 7SK sn...,-,...,-,regulatory,regulatory,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000213639,"[GO:0070062, GO:0032922, GO:0000165, GO:000078..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479697,970575,ncRI-40970577,GAS5,NONHSAG003504,lncRNA,TRAIL,P50591,protein,Pi-sno75 significantly induces epigenetic alte...,RIP,...,MCF7,expression correlation,expression correlation,RNA-Protein,Literature Mining,URS00008C4009,ENSG00000234741,NaN,ENSG00000121858,"[GO:0070062, GO:0005102, GO:0042802, GO:000551..."
479698,970575,ncRI-40970577,GAS5,NONHSAG003504,lncRNA,TRAIL,P50591,protein,Pi-sno75 significantly induces epigenetic alte...,RIP,...,MCF7,expression correlation,expression correlation,RNA-Protein,Literature Mining,URS00009B629F,ENSG00000234741,NaN,ENSG00000121858,"[GO:0070062, GO:0005102, GO:0042802, GO:000551..."
479699,970575,ncRI-40970577,GAS5,NONHSAG003504,lncRNA,TRAIL,P50591,protein,Pi-sno75 significantly induces epigenetic alte...,RIP,...,MCF7,expression correlation,expression correlation,RNA-Protein,Literature Mining,URS00009BCE1D,ENSG00000234741,NaN,ENSG00000121858,"[GO:0070062, GO:0005102, GO:0042802, GO:000551..."
479700,970575,ncRI-40970577,GAS5,NONHSAG003504,lncRNA,TRAIL,P50591,protein,Pi-sno75 significantly induces epigenetic alte...,RIP,...,MCF7,expression correlation,expression correlation,RNA-Protein,Literature Mining,URS00009C227F,ENSG00000234741,NaN,ENSG00000121858,"[GO:0070062, GO:0005102, GO:0042802, GO:000551..."


In [99]:
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2.explode('go_terms')
npinterv4_tarid_go_terms_v2

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0106140
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0004860
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0042802
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0007507
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0017069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,970578,ncRI-40970580,SNORD50B,ENSG00000275072,snoRNA,KRAS,P01116,protein,SNORD50A and SNORD50B snoRNAs thus directly bi...,"RIP,Luciferase reporter assay",...,-,ncRNA-protein binding,binding,RNA-Protein,Literature Mining,URS00006F8A66,ENSG00000275072,NaN,ENSG00000133703,GO:0043524
479701,970578,ncRI-40970580,SNORD50B,ENSG00000275072,snoRNA,KRAS,P01116,protein,SNORD50A and SNORD50B snoRNAs thus directly bi...,"RIP,Luciferase reporter assay",...,-,ncRNA-protein binding,binding,RNA-Protein,Literature Mining,URS00006F8A66,ENSG00000275072,NaN,ENSG00000133703,GO:0019003
479701,970578,ncRI-40970580,SNORD50B,ENSG00000275072,snoRNA,KRAS,P01116,protein,SNORD50A and SNORD50B snoRNAs thus directly bi...,"RIP,Luciferase reporter assay",...,-,ncRNA-protein binding,binding,RNA-Protein,Literature Mining,URS00006F8A66,ENSG00000275072,NaN,ENSG00000133703,GO:0060441
479701,970578,ncRI-40970580,SNORD50B,ENSG00000275072,snoRNA,KRAS,P01116,protein,SNORD50A and SNORD50B snoRNAs thus directly bi...,"RIP,Luciferase reporter assay",...,-,ncRNA-protein binding,binding,RNA-Protein,Literature Mining,URS00006F8A66,ENSG00000275072,NaN,ENSG00000133703,GO:0007265


In [101]:
len(np.unique(npinterv4_tarid_go_terms_v2["go_terms"]))

5497

In [102]:
for go_term in np.unique(npinterv4_tarid_go_terms_v2["go_terms"]):
    try:
        obodag[go_term]
    except KeyError:
        print(go_term)

GO:0003925
GO:0062152
GO:0062153
GO:0062176
GO:0106226
GO:0106230
GO:0106231
GO:0106256
GO:0106310
GO:0106312
GO:0106313
GO:0106363
GO:0120223
GO:0120259
GO:0120283
GO:0120293
GO:0120301
GO:0140416
GO:0140431
GO:0140447
GO:0140448
GO:0140492
GO:0140517
GO:0140545
GO:0140585
GO:0140658
GO:0140668
GO:0140672
GO:0140681
GO:0140682
GO:0140684


In [81]:
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0120259']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140545']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140585']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140658']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140668']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140672']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140681']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140682']
npinterv4_tarid_go_terms_v2 = npinterv4_tarid_go_terms_v2[npinterv4_tarid_go_terms_v2["go_terms"] != 'GO:0140684']
npinterv4_tarid_go_terms_v2

,index,interID,ncName,ncID,ncType,tarName,tarID,tarType,interDescription,experiment,...,tissueOrCell,tag,class,level,datasource,ncID_rnacentral_code,ncID_ensembl_code,tarID_rnacentral_code,tarID_ensembl_code,go_terms
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0106140
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:1901798
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0004860
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0042802
0,5,ncRI-40000006,7SK,NONHSAG040596,lncRNA,HEXIM1,O94992,protein,Binding of the 7SK snRNA turns the HEXIM1 prot...,Yeast two-hybrid analysis;Immunofluorescence;M...,...,-,ncRNA-protein binding,binding,RNA-Protein,Literature mining,URS00004E902D,ENSG00000249352,NaN,ENSG00000186834,GO:0007507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035522
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0035455
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0006995
487274,1100455,ncRI-41100457,SNORD43,ENSG00000263764,snoRNA,SNHG1,ENSG00000255717,ncRNA,RNA-RNA interaction from RISE database,LIGRseq,...,HEK293T,RNA-RNA interaction,binding,RNA-RNA,RISE database,URS000063F175,ENSG00000263764,URS0000EF6DD5,ENSG00000255717,GO:0005852


##### Save file

In [142]:
npinterv4_tarid_go_terms_df_v2 = pd.DataFrame({
    0: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    1: npinterv4_tarid_go_terms_v2["tarID_ensembl_code"],
    2: list(npinterv4_tarid_go_terms_v2.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    3: ["enables"]  * len(npinterv4_tarid_go_terms_v2),
    4: npinterv4_tarid_go_terms_v2["go_terms"],
    5: ["GO_REF:0000001"] * len(npinterv4_tarid_go_terms_v2),
    6: ["EXP"] * len(npinterv4_tarid_go_terms_v2),
    7: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    8: [np.nan if go_term not in list(obodag.keys()) else "P" if obodag[go_term].namespace == "biological_process" else "F" if obodag[go_term].namespace == "molecular_function" else "C" for go_term in npinterv4_tarid_go_terms_v2["go_terms"]], 
    9: [np.nan if go_term not in list(obodag.keys()) else obodag[go_term].name for go_term in npinterv4_tarid_go_terms_v2["go_terms"]],
    10: list(npinterv4_tarid_go_terms_v2.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    11: ["ncRNA"] * len(npinterv4_tarid_go_terms_v2),
    12: ["taxon:9606"] * len(npinterv4_tarid_go_terms_v2),
    13: [20220215] * len(npinterv4_tarid_go_terms_v2),
    14: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    15: [np.nan] * len(npinterv4_tarid_go_terms_v2),
    16: [np.nan] * len(npinterv4_tarid_go_terms_v2)
})

npinterv4_tarid_go_terms_df_v2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0106140,GO_REF:0000001,EXP,NPInter,F,P-TEFb complex binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0004860,GO_REF:0000001,EXP,NPInter,F,protein kinase inhibitor activity,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0042802,GO_REF:0000001,EXP,NPInter,F,identical protein binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0007507,GO_REF:0000001,EXP,NPInter,P,heart development,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0017069,GO_REF:0000001,EXP,NPInter,F,snRNA binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0043524,GO_REF:0000001,EXP,NPInter,P,negative regulation of neuron apoptotic process,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0019003,GO_REF:0000001,EXP,NPInter,F,GDP binding,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0060441,GO_REF:0000001,EXP,NPInter,P,epithelial tube branching involved in lung mor...,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0007265,GO_REF:0000001,EXP,NPInter,P,Ras protein signal transduction,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [106]:
npinterv4_tarid_go_terms_df_v2 = pd.DataFrame({
    0: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    1: npinterv4_tarid_go_terms_v2["tarID_ensembl_code"],
    2: list(npinterv4_tarid_go_terms_v2.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    3: ["enables"]  * len(npinterv4_tarid_go_terms_v2),
    4: npinterv4_tarid_go_terms_v2["go_terms"],
    5: ["GO_REF:0000001"] * len(npinterv4_tarid_go_terms_v2),
    6: ["EXP"] * len(npinterv4_tarid_go_terms_v2),
    7: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    8: [np.nan if go_term not in list(obodag.keys()) else "P" if obodag[go_term].namespace == "biological_process" else "F" if obodag[go_term].namespace == "molecular_function" else "C" for go_term in npinterv4_tarid_go_terms_v2["go_terms"]], 
    9: [np.nan if go_term not in list(obodag.keys()) else obodag[go_term].name for go_term in npinterv4_tarid_go_terms_v2["go_terms"]],
    10: list(npinterv4_tarid_go_terms_v2.join(ensembl_gene_gene_name.set_index("Gene stable ID"), on="tarID_ensembl_code")["Gene name"].astype("str")),
    11: ["ncRNA"] * len(npinterv4_tarid_go_terms_v2),
    12: ["taxon:9606"] * len(npinterv4_tarid_go_terms_v2),
    13: [20220215] * len(npinterv4_tarid_go_terms_v2),
    14: ["NPInter"] * len(npinterv4_tarid_go_terms_v2),
    15: [np.nan] * len(npinterv4_tarid_go_terms_v2),
    16: [np.nan] * len(npinterv4_tarid_go_terms_v2)
})

npinterv4_tarid_go_terms_df_v2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0106140,NPInter,EXP,NPInter,F,P-TEFb complex binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0004860,NPInter,EXP,NPInter,F,protein kinase inhibitor activity,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0042802,NPInter,EXP,NPInter,F,identical protein binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0007507,NPInter,EXP,NPInter,P,heart development,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0017069,NPInter,EXP,NPInter,F,snRNA binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0043524,NPInter,EXP,NPInter,P,negative regulation of neuron apoptotic process,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0019003,NPInter,EXP,NPInter,F,GDP binding,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0060441,NPInter,EXP,NPInter,P,epithelial tube branching involved in lung mor...,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0007265,NPInter,EXP,NPInter,P,Ras protein signal transduction,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN


In [143]:
npinterv4_tarid_go_terms_df_v2 = npinterv4_tarid_go_terms_df_v2[~npinterv4_tarid_go_terms_df_v2[8].isnull()]
npinterv4_tarid_go_terms_df_v2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0106140,GO_REF:0000001,EXP,NPInter,F,P-TEFb complex binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0004860,GO_REF:0000001,EXP,NPInter,F,protein kinase inhibitor activity,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0042802,GO_REF:0000001,EXP,NPInter,F,identical protein binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0007507,GO_REF:0000001,EXP,NPInter,P,heart development,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0017069,GO_REF:0000001,EXP,NPInter,F,snRNA binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0043524,GO_REF:0000001,EXP,NPInter,P,negative regulation of neuron apoptotic process,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0019003,GO_REF:0000001,EXP,NPInter,F,GDP binding,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0060441,GO_REF:0000001,EXP,NPInter,P,epithelial tube branching involved in lung mor...,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0007265,GO_REF:0000001,EXP,NPInter,P,Ras protein signal transduction,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [107]:
npinterv4_tarid_go_terms_df_v2 = npinterv4_tarid_go_terms_df_v2[~npinterv4_tarid_go_terms_df_v2[8].isnull()]
npinterv4_tarid_go_terms_df_v2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0106140,NPInter,EXP,NPInter,F,P-TEFb complex binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0004860,NPInter,EXP,NPInter,F,protein kinase inhibitor activity,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0042802,NPInter,EXP,NPInter,F,identical protein binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0007507,NPInter,EXP,NPInter,P,heart development,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0017069,NPInter,EXP,NPInter,F,snRNA binding,HEXIM1,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0043524,NPInter,EXP,NPInter,P,negative regulation of neuron apoptotic process,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0019003,NPInter,EXP,NPInter,F,GDP binding,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0060441,NPInter,EXP,NPInter,P,epithelial tube branching involved in lung mor...,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0007265,NPInter,EXP,NPInter,P,Ras protein signal transduction,KRAS,ncRNA,taxon:9606,NPInter,NPInter,NaN,NaN


In [144]:
from itertools import islice
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human.gaf', "r") as myfile:
    head = list(islice(myfile, 41))
head

['!gaf-version: 2.2\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T16:45\n',
 '!\n',
 '!Header from source association file:\n',
 '!=================================\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T04:41\n',
 '!\n',
 '!Header from goa_human source association file:\n',
 '!=================================\n',
 '!\n',
 '!The set of protein accessions included in this file is based on UniProt reference proteomes, which provide one protein per gene.\n',
 '!They include the protein sequences annotated in Swiss-Prot or the longest TrEMBL transcript if there is no Swiss-Prot record.\n',
 '!If a particular protein accession is not annotated with GO, then it will not appear in this file.\n',
 '!\n',
 '!Note that the annotation set in this file is filtered in order to reduce redundancy; the full, unfiltered set can be found in\n',
 '!ftp://ftp.ebi.ac.uk/pub/databases/GO/goa/UNIPROT/goa_uniprot_all.gz\n',
 '!\n',
 '!date-gene

In [145]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_v2_ensembl_gene_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [146]:
npinterv4_tarid_go_terms_df_v2.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_v2_ensembl_gene_mapping.gaf', mode="a", sep="\t", header=False, index=False)

## Create Transcript NPInter versions

In [149]:
from pybiomart import Server

server = Server(host='http://www.ensembl.org')

dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
                 .datasets['hsapiens_gene_ensembl'])

In [151]:
ensembl_gene_ensembl_transcript = dataset.query(attributes=['ensembl_gene_id', 'ensembl_transcript_id'])
ensembl_gene_ensembl_transcript

,Gene stable ID,Transcript stable ID
0,ENSG00000210049,ENST00000387314
1,ENSG00000211459,ENST00000389680
2,ENSG00000210077,ENST00000387342
3,ENSG00000210082,ENST00000387347
4,ENSG00000209082,ENST00000386347
...,...,...
266610,ENSG00000086619,ENST00000689343
266611,ENSG00000086619,ENST00000691519
266612,ENSG00000086619,ENST00000686625
266613,ENSG00000086619,ENST00000366589


In [157]:
npinterv4_ncid_go_terms_transcript_df = npinterv4_ncid_go_terms_df.copy()
npinterv4_ncid_go_terms_transcript_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0010468,GO_REF:0000001,EXP,NPInter,P,regulation of gene expression,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0040033,GO_REF:0000001,EXP,NPInter,P,"negative regulation of translation, ncRNA-medi...",LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENSG00000249352,LINC02198,enables,GO:0005515,GO_REF:0000001,EXP,NPInter,F,protein binding,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100456,NPInter,ENSG00000208892,SNORA49,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,SNORA49,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENSG00000200753,nan,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENSG00000200753,nan,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100515,NPInter,ENSG00000283146,MIR548AU,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,MIR548AU,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [156]:
npinterv4_tarid_go_terms_transcript_df_v2 = npinterv4_tarid_go_terms_df_v2.copy()
npinterv4_tarid_go_terms_transcript_df_v2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0106140,GO_REF:0000001,EXP,NPInter,F,P-TEFb complex binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0004860,GO_REF:0000001,EXP,NPInter,F,protein kinase inhibitor activity,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0042802,GO_REF:0000001,EXP,NPInter,F,identical protein binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0007507,GO_REF:0000001,EXP,NPInter,P,heart development,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
0,NPInter,ENSG00000186834,HEXIM1,enables,GO:0017069,GO_REF:0000001,EXP,NPInter,F,snRNA binding,HEXIM1,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0043524,GO_REF:0000001,EXP,NPInter,P,negative regulation of neuron apoptotic process,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0019003,GO_REF:0000001,EXP,NPInter,F,GDP binding,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0060441,GO_REF:0000001,EXP,NPInter,P,epithelial tube branching involved in lung mor...,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
479701,NPInter,ENSG00000133703,KRAS,enables,GO:0007265,GO_REF:0000001,EXP,NPInter,P,Ras protein signal transduction,KRAS,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [ ]:
######

In [158]:
npinterv4_ncid_go_terms_transcript_df = npinterv4_ncid_go_terms_transcript_df.join(ensembl_gene_ensembl_transcript.set_index("Gene stable ID"), on=[1])
npinterv4_ncid_go_terms_transcript_df = npinterv4_ncid_go_terms_transcript_df[~npinterv4_ncid_go_terms_transcript_df["Transcript stable ID"].isnull()]
npinterv4_ncid_go_terms_transcript_df[1] = npinterv4_ncid_go_terms_transcript_df["Transcript stable ID"]
del npinterv4_ncid_go_terms_transcript_df["Transcript stable ID"]
npinterv4_ncid_go_terms_transcript_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
3,NPInter,ENST00000505371,LINC02198,enables,GO:0010468,GO_REF:0000001,EXP,NPInter,P,regulation of gene expression,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENST00000504280,LINC02198,enables,GO:0010468,GO_REF:0000001,EXP,NPInter,P,regulation of gene expression,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENST00000505371,LINC02198,enables,GO:0040033,GO_REF:0000001,EXP,NPInter,P,"negative regulation of translation, ncRNA-medi...",LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENST00000504280,LINC02198,enables,GO:0040033,GO_REF:0000001,EXP,NPInter,P,"negative regulation of translation, ncRNA-medi...",LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
3,NPInter,ENST00000505371,LINC02198,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,LINC02198,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100456,NPInter,ENST00000386157,SNORA49,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,SNORA49,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENST00000363883,nan,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100460,NPInter,ENST00000363883,nan,enables,GO:0034660,GO_REF:0000001,EXP,NPInter,P,ncRNA metabolic process,nan,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN
1100515,NPInter,ENST00000637016,MIR548AU,enables,GO:0003723,GO_REF:0000001,EXP,NPInter,F,RNA binding,MIR548AU,ncRNA,taxon:9606,20220215,NPInter,NaN,NaN


In [ ]:
npinterv4_tarid_go_terms_transcript_df_v2 = npinterv4_tarid_go_terms_transcript_df_v2.join(ensembl_gene_ensembl_transcript.set_index("Gene stable ID"), on=[1])
npinterv4_tarid_go_terms_transcript_df_v2 = npinterv4_tarid_go_terms_transcript_df_v2[~npinterv4_tarid_go_terms_transcript_df_v2["Transcript stable ID"].isnull()]
npinterv4_tarid_go_terms_transcript_df_v2[1] = npinterv4_tarid_go_terms_transcript_df_v2["Transcript stable ID"]
del npinterv4_tarid_go_terms_transcript_df_v2["Transcript stable ID"]
npinterv4_tarid_go_terms_transcript_df_v2

In [160]:
from itertools import islice
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human.gaf', "r") as myfile:
    head = list(islice(myfile, 41))
head

['!gaf-version: 2.2\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T16:45\n',
 '!\n',
 '!Header from source association file:\n',
 '!=================================\n',
 '!\n',
 '!generated-by: GOC\n',
 '!\n',
 '!date-generated: 2022-01-14T04:41\n',
 '!\n',
 '!Header from goa_human source association file:\n',
 '!=================================\n',
 '!\n',
 '!The set of protein accessions included in this file is based on UniProt reference proteomes, which provide one protein per gene.\n',
 '!They include the protein sequences annotated in Swiss-Prot or the longest TrEMBL transcript if there is no Swiss-Prot record.\n',
 '!If a particular protein accession is not annotated with GO, then it will not appear in this file.\n',
 '!\n',
 '!Note that the annotation set in this file is filtered in order to reduce redundancy; the full, unfiltered set can be found in\n',
 '!ftp://ftp.ebi.ac.uk/pub/databases/GO/goa/UNIPROT/goa_uniprot_all.gz\n',
 '!\n',
 '!date-gene

In [161]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_nc_ensembl_transcript_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [162]:
npinterv4_ncid_go_terms_transcript_df.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_nc_ensembl_transcript_mapping.gaf', mode="a", sep="\t", header=False, index=False)

In [145]:
# always remember, use files in a with statement
with open('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_v2_ensembl_transcript_mapping.gaf', "w") as f2:
    for item in head:
        f2.write(item)

In [146]:
npinterv4_tarid_go_terms_transcript_df_v2.to_csv('/dh-projects/uk_bb_intergenics/analysis/development/hollmann/data/agnostic/goa_human_npinter_tar_v2_ensembl_transcript_mapping.gaf', mode="a", sep="\t", header=False, index=False)